# AoU Cost Engine — Demo

This notebook demonstrates the AoU Cost Engine's key features:
1. **Cost estimation** via BigQuery dry run
2. **Fallback estimation** using the static CDR catalog
3. **Guardrails** — cost thresholds and byte cap suggestions
4. **AI optimization** — Claude-powered query rewrites

## Setup

On the AoU Workbench, the BigQuery client is already authenticated. Just load the extension:

In [ ]:
%load_ext aou_cost_engine

## 1. Basic Cost Estimation (Dry Run)

Add `%%aou_cost` to any cell to see its cost before running it.
When a BigQuery client is available, this uses **dry run** for exact numbers.

In [ ]:
%%aou_cost
SELECT person_id, condition_concept_id, condition_start_date
FROM `condition_occurrence`
WHERE condition_start_date > '2020-01-01'

## 2. Expensive Query Warning

The engine flags costly patterns like `SELECT *` and suggests byte caps.

In [ ]:
%%aou_cost
SELECT *
FROM `cb_variant_to_person`
LIMIT 1000

## 3. Fallback Mode (No BigQuery Client)

Use `--fallback` to estimate from the static catalog when dry run isn't available.
Results are clearly labeled as **approximate**.

In [ ]:
%%aou_cost --fallback
SELECT person_id, measurement_concept_id, value_as_number
FROM `measurement`
WHERE measurement_date > '2021-01-01'

## 4. AI Optimization

Enable AI to get Claude-powered query rewrites with verified savings.

In [ ]:
# Enable AI optimization globally
%aou_cost_config --ai on --threshold 0.05

In [ ]:
%%aou_cost --ai
SELECT *
FROM `measurement`
WHERE measurement_concept_id = 3004249

## 5. Programmatic Usage

You can also use the engine programmatically without the magic:

In [ ]:
from aou_cost_engine import estimate_bq_cost, classify_cell, CellType
from google.cloud import bigquery

client = bigquery.Client()

sql = "SELECT person_id FROM person"
estimate = estimate_bq_cost(sql, client)

print(f"Bytes scanned: {estimate.bytes_display}")
print(f"Estimated cost: {estimate.cost_display}")
print(f"Exact: {estimate.exact}")
print(f"Cache eligible: {estimate.cache_eligible}")

In [ ]:
# Use the fallback estimator without a BigQuery client
from aou_cost_engine import estimate_from_sql

estimate = estimate_from_sql("SELECT * FROM cb_variant_to_person")
print(f"Approximate bytes: {estimate.bytes_display}")
print(f"Approximate cost: {estimate.cost_display}")
for w in estimate.warnings:
    print(f"  ⚠ {w}")